# 🎬 Easy-Edits AutoCrop Worker

This notebook acts as a GPU worker that processes video autocrop jobs using **Cinematic Camera Modes** and **Whisper Transcription**:
- **Tripod Mode**: Static camera when subject fits in frame (prevents jitter)
- **Dolly Mode**: Smooth panning when subject moves too far
- **Whisper AI**: Automatic transcription with word-level timestamps
- **Karaoke Captions**: Word-by-word highlighted captions

**Instructions:**
1. Make sure GPU runtime is enabled (Runtime → Change runtime type → GPU)
2. Run each cell in order (1-7)
3. Run cell 8 to start the worker loop

In [ ]:
# Cell 1: Install Dependencies
%pip install -q ultralytics opencv-python firebase-admin boto3 scenedetect tqdm numpy openai-whisper

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Configure Credentials
import json

from google.colab import userdata

# Credentials are read from Colab's Secrets panel (key icon in the left sidebar).
# Add these secrets and grant this notebook access to them:
#   FIREBASE_CREDENTIALS  - the full Firebase service-account JSON
#   R2_ENDPOINT           - https://<account-id>.r2.cloudflarestorage.com
#   R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_BUCKET_NAME
FIREBASE_CREDENTIALS = userdata.get('FIREBASE_CREDENTIALS')

R2_CONFIG = {
    "endpoint": userdata.get('R2_ENDPOINT'),
    "access_key_id": userdata.get('R2_ACCESS_KEY_ID'),
    "secret_access_key": userdata.get('R2_SECRET_ACCESS_KEY'),
    "bucket_name": userdata.get('R2_BUCKET_NAME'),
}

firebase_creds = json.loads(FIREBASE_CREDENTIALS)
print("✅ Credentials configured")

In [ ]:
# Cell 3: Initialize Services
import firebase_admin
from firebase_admin import credentials, firestore
import boto3
from botocore.config import Config

if not firebase_admin._apps:
    cred = credentials.Certificate(firebase_creds)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase Firestore connected")

r2_client = boto3.client(
    's3',
    endpoint_url=R2_CONFIG['endpoint'],
    aws_access_key_id=R2_CONFIG['access_key_id'],
    aws_secret_access_key=R2_CONFIG['secret_access_key'],
    config=Config(signature_version='s3v4'),
    region_name='auto'
)

try:
    r2_client.list_objects_v2(Bucket=R2_CONFIG['bucket_name'], MaxKeys=1)
    print("✅ Cloudflare R2 connected")
except Exception as e:
    print(f"❌ R2 connection failed: {e}")

In [ ]:
# Cell 4: Cinematic AutoCrop Functions (Tripod/Dolly Modes)
import cv2
import scenedetect
import subprocess
from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector
from ultralytics import YOLO
import numpy as np
import os
from tqdm import tqdm

# --- Configuration ---
ASPECT_RATIO = 9 / 16
YOLO_MODEL = 'yolov8m.pt'  # Medium model for stable detection
SMOOTHING_WINDOW_SIZE = 60  # 60 frames = 2 sec @ 30fps

model = YOLO(YOLO_MODEL)
print(f"✅ YOLO model loaded: {YOLO_MODEL} on {model.device}")

def get_video_props(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return w, h, fps, count

def get_dominant_subject_x(frame):
    """Returns center X of largest person in frame, or None."""
    results = model([frame], verbose=False)
    best_area = 0
    best_center = None
    
    for result in results:
        for box in result.boxes:
            if box.cls[0] == 0:  # Person
                x1, y1, x2, y2 = [int(i) for i in box.xyxy[0]]
                area = (x2 - x1) * (y2 - y1)
                if area > best_area:
                    best_area = area
                    best_center = (x1 + x2) / 2
    return best_center

def generate_camera_path(video_path, start_frame, end_frame, original_width, crop_width):
    """Tripod Mode (static) or Dolly Mode (smooth pan) based on motion range."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    raw_centers = []
    num_frames = end_frame - start_frame
    
    for _ in range(num_frames):
        ret, frame = cap.read()
        if not ret: break
        
        center = get_dominant_subject_x(frame)
        if center is not None:
            raw_centers.append(center)
        else:
            raw_centers.append(raw_centers[-1] if raw_centers else original_width/2)
            
    cap.release()
    
    # If no frames read, return center for all
    if not raw_centers:
        return [original_width / 2] * max(1, num_frames)

    min_x, max_x = min(raw_centers), max(raw_centers)
    motion_range = max_x - min_x
    safe_zone_width = crop_width * 0.9
    
    # TRIPOD MODE: Subject fits in one static frame
    if motion_range < safe_zone_width:
        avg_center = np.mean(raw_centers)
        final_path = [avg_center] * len(raw_centers)
    # DOLLY MODE: Must pan, apply heavy smoothing
    else:
        window = np.ones(SMOOTHING_WINDOW_SIZE) / SMOOTHING_WINDOW_SIZE
        final_path = np.convolve(raw_centers, window, mode='same')
        # Fix edge artifacts
        blend_length = min(20, len(final_path))
        if blend_length > 0:
            for i in range(blend_length):
                final_path[i] = raw_centers[i]
                final_path[-(i+1)] = raw_centers[-(i+1)]

    return list(final_path)

def detect_scenes(video_path, fps):
    """Detect scene cuts to reset camera per shot."""
    video_manager = VideoManager([video_path])
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=27.0))
    video_manager.set_downscale_factor()
    video_manager.start()
    scene_manager.detect_scenes(frame_source=video_manager)
    scene_list = scene_manager.get_scene_list()
    video_manager.release()
    return scene_list

def process_video(input_path, output_path, progress_callback=None):
    """Main cinematic autocrop function."""
    print(f"🎬 Processing: {input_path}")
    
    # Temp files
    base_name = os.path.splitext(output_path)[0]
    temp_video = f"{base_name}_temp_video.mp4"
    temp_audio = f"{base_name}_temp_audio.aac"
    
    # Get video properties
    orig_w, orig_h, fps, total_frames = get_video_props(input_path)
    out_h = orig_h
    out_w = int(out_h * ASPECT_RATIO)
    if out_w % 2 != 0: out_w += 1
    
    print(f"   Source: {orig_w}x{orig_h} → Target: {out_w}x{out_h}")
    
    # Detect scenes
    print("🔍 Step 1: Detecting scenes...")
    scenes = detect_scenes(input_path, fps)
    
    # IMPORTANT: If no scenes detected, treat entire video as one scene
    if not scenes:
        print("   No scene cuts detected, treating as single scene")
        scenes = [(scenedetect.FrameTimecode(0, fps), scenedetect.FrameTimecode(total_frames, fps))]
    else:
        print(f"   Found {len(scenes)} scene(s)")
    
    # Calculate camera paths per scene
    print("🧠 Step 2: Calculating cinematic paths...")
    frame_map = {}
    for start_tc, end_tc in tqdm(scenes, desc="Analyzing"):
        start_f = start_tc.get_frames()
        end_f = end_tc.get_frames()
        path = generate_camera_path(input_path, start_f, end_f, orig_w, out_w)
        for i, center_x in enumerate(path):
            frame_map[start_f + i] = center_x
    
    # Render cropped video
    print("✂️ Step 3: Rendering...")
    # IMPORTANT: -pix_fmt yuv420p is required for browser compatibility!
    cmd = [
        'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
        '-s', f'{out_w}x{out_h}', '-pix_fmt', 'bgr24',
        '-r', str(fps), '-i', '-', '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',  # Browser compatibility fix!
        '-preset', 'fast', '-crf', '18', '-an', temp_video
    ]
    
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
    cap = cv2.VideoCapture(input_path)
    
    for f_idx in tqdm(range(total_frames), desc="Encoding"):
        ret, frame = cap.read()
        if not ret: break
        
        # Use pre-calculated center, default to center if not found
        center_x = frame_map.get(f_idx, orig_w / 2)
        x1 = int(center_x - out_w / 2)
        if x1 < 0: x1 = 0
        if x1 + out_w > orig_w: x1 = orig_w - out_w
        x2 = x1 + out_w
        
        crop = frame[0:out_h, x1:x2]
        proc.stdin.write(crop.tobytes())
        
        if progress_callback and f_idx % 30 == 0:
            progress_callback(int((f_idx / total_frames) * 100))
    
    proc.stdin.close()
    proc.wait()
    cap.release()
    
    # Finalize with audio
    print("🔊 Step 4: Adding audio...")
    subprocess.run(['ffmpeg', '-y', '-i', input_path, '-vn', '-acodec', 'aac', '-b:a', '128k', temp_audio], 
                   stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    subprocess.run(['ffmpeg', '-y', '-i', temp_video, '-i', temp_audio, '-c:v', 'copy', '-c:a', 'aac', output_path], 
                   stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    
    # Cleanup
    if os.path.exists(temp_video): os.remove(temp_video)
    if os.path.exists(temp_audio): os.remove(temp_audio)
    
    print(f"✅ Done: {output_path}")
    return output_path

print("✅ Cinematic AutoCrop functions loaded (Tripod/Dolly modes)")

In [ ]:
# Cell 5: Whisper Transcription Functions
import whisper

# Load Whisper model (base for speed, use 'small' or 'medium' for better accuracy)
WHISPER_MODEL = 'base'
whisper_model = whisper.load_model(WHISPER_MODEL)
print(f"✅ Whisper model loaded: {WHISPER_MODEL}")

def transcribe_video(video_path):
    """
    Transcribe video using Whisper with word-level timestamps.
    Returns list of words with start/end times.
    """
    print(f"🎤 Transcribing: {video_path}")
    
    # Transcribe with word timestamps
    result = whisper_model.transcribe(
        video_path,
        word_timestamps=True,
        verbose=False
    )
    
    # Extract word-level data
    words = []
    for segment in result.get('segments', []):
        for word_data in segment.get('words', []):
            words.append({
                'word': word_data['word'].strip(),
                'start': round(word_data['start'], 3),
                'end': round(word_data['end'], 3)
            })
    
    print(f"   Found {len(words)} words")
    print(f"   Full text: {result.get('text', '')[:100]}...")
    
    return {
        'words': words,
        'text': result.get('text', ''),
        'language': result.get('language', 'en')
    }

print("✅ Whisper transcription functions loaded")

In [ ]:
# Cell 6: Caption Generation Functions (Karaoke-style)
import re

# Theme configurations
FONT_THEMES = {
    'montserrat': 'Montserrat',
    'poppins': 'Poppins',
    'roboto': 'Roboto',
    'opensans': 'Open Sans',
    'inter': 'Inter',
    'bebas': 'Bebas Neue',
    'oswald': 'Oswald',
    'anton': 'Anton'
}

COLOR_THEMES = {
    'white': '&HFFFFFF',
    'black': '&H000000',
    'yellow': '&H00FFFF',
    'red': '&H0000FF',
    'cyan': '&HFFFF00',
    'lime': '&H00FF00',
    'pink': '&HFF00FF',
    'orange': '&H0080FF'
}

HIGHLIGHT_THEMES = {
    'yellow': '&H00FFFF',
    'cyan': '&HFFFF00',
    'lime': '&H00FF00',
    'magenta': '&HFF00FF',
    'orange': '&H0080FF',
    'red': '&H0000FF',
    'white': '&HFFFFFF',
    'gold': '&H00D7FF'
}

def format_ass_time(seconds):
    """Convert seconds to ASS time format (H:MM:SS.cc)"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f"{hours}:{minutes:02d}:{secs:05.2f}"

def generate_ass_subtitles(transcription, output_path, settings):
    """
    Generate ASS subtitle file with karaoke-style word highlighting.
    
    settings:
        font: font key from FONT_THEMES
        color: color key from COLOR_THEMES
        highlightColor: color key from HIGHLIGHT_THEMES
        wordsPerSegment: number of words per caption line (default 4)
    """
    words = transcription.get('words', [])
    if not words:
        print("⚠️ No words to generate captions")
        return None
    
    font = FONT_THEMES.get(settings.get('font', 'montserrat'), 'Montserrat')
    color = COLOR_THEMES.get(settings.get('color', 'white'), '&HFFFFFF')
    highlight = HIGHLIGHT_THEMES.get(settings.get('highlightColor', 'yellow'), '&H00FFFF')
    words_per_segment = settings.get('wordsPerSegment', 4)
    
    # ASS header
    ass_content = f"""[Script Info]
Title: Easy-Edits Captions
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920
WrapStyle: 0

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{font},80,{color},&H000000FF,&H00000000,&H80000000,1,0,0,0,100,100,0,0,1,4,2,2,40,40,120,1
Style: Highlight,{font},80,{highlight},&H000000FF,&H00000000,&H80000000,1,0,0,0,100,100,0,0,1,4,2,2,40,40,120,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    
    # Group words into segments
    segments = []
    for i in range(0, len(words), words_per_segment):
        segment_words = words[i:i + words_per_segment]
        if segment_words:
            segments.append({
                'words': segment_words,
                'start': segment_words[0]['start'],
                'end': segment_words[-1]['end']
            })
    
    # Generate karaoke lines
    for segment in segments:
        start_time = format_ass_time(segment['start'])
        end_time = format_ass_time(segment['end'])
        
        # Build karaoke text with word-by-word highlighting
        karaoke_text = ""
        segment_start = segment['start']
        
        for word_data in segment['words']:
            word = word_data['word']
            word_start = word_data['start']
            word_end = word_data['end']
            word_duration = int((word_end - word_start) * 100)  # centiseconds
            pre_delay = int((word_start - segment_start) * 100)  # time before this word
            
            # Use \k for karaoke timing (highlights word when spoken)
            if pre_delay > 0:
                karaoke_text += f"{{\\k{pre_delay}}}"
            karaoke_text += f"{{\\kf{word_duration}}}{word} "
            segment_start = word_end
        
        ass_content += f"Dialogue: 0,{start_time},{end_time},Default,,0,0,0,,{karaoke_text.strip()}\n"
    
    # Write ASS file
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(ass_content)
    
    print(f"✅ Generated ASS subtitles: {output_path}")
    return output_path

def burn_captions(input_video, output_video, ass_file):
    """
    Burn ASS subtitles into video using FFmpeg.
    """
    print(f"🔥 Burning captions into video...")
    
    cmd = [
        'ffmpeg', '-y',
        '-i', input_video,
        '-vf', f"ass='{ass_file}'",
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        '-preset', 'fast',
        '-crf', '18',
        '-c:a', 'copy',
        output_video
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ FFmpeg error: {result.stderr}")
        raise Exception(f"Caption burn failed: {result.stderr}")
    
    print(f"✅ Captioned video: {output_video}")
    return output_video

print("✅ Caption generation functions loaded (Karaoke-style)")

In [ ]:
# Cell 7: Worker Functions
import time
from google.cloud.firestore_v1.base_query import FieldFilter

def download_from_r2(key, local_path):
    print(f"⬇️ Downloading: {key}")
    r2_client.download_file(R2_CONFIG['bucket_name'], key, local_path)
    print(f"✅ Downloaded")

def upload_to_r2(local_path, key):
    print(f"⬆️ Uploading: {key}")
    r2_client.upload_file(local_path, R2_CONFIG['bucket_name'], key)
    print(f"✅ Uploaded")

def update_job_status(job_id, status, **kwargs):
    db.collection('jobs').document(job_id).update({'status': status, 'updatedAt': firestore.SERVER_TIMESTAMP, **kwargs})
    print(f"📝 Job {job_id[:8]}... → {status}")

def get_pending_jobs():
    jobs = db.collection('jobs').where(filter=FieldFilter('status', '==', 'pending')).limit(5).get()
    return [job.to_dict() for job in jobs]

def get_caption_jobs():
    """Get jobs waiting for caption generation."""
    jobs = db.collection('jobs').where(filter=FieldFilter('status', '==', 'caption_pending')).limit(5).get()
    return [job.to_dict() for job in jobs]

def process_job(job):
    """Process autocrop job with transcription."""
    job_id = job['id']
    input_key = job['inputVideoKey']
    print(f"\n{'='*50}\n🎬 Processing job: {job_id}\n{'='*50}")
    try:
        update_job_status(job_id, 'processing')
        os.makedirs('/content/temp', exist_ok=True)
        
        # Download input video
        input_path = f'/content/temp/input_{job_id}.mp4'
        download_from_r2(input_key, input_path)
        
        # Step 1: Autocrop video
        output_path = f'/content/temp/output_{job_id}.mp4'
        process_video(input_path, output_path, lambda p: update_job_status(job_id, 'processing', progress=p))
        
        # Step 2: Transcribe the cropped video
        print("\n🎤 Starting transcription...")
        update_job_status(job_id, 'transcribing', progress=90)
        transcription = transcribe_video(output_path)
        
        # Step 3: Upload cropped video
        output_key = f"outputs/cropped_{job_id}.mp4"
        upload_to_r2(output_path, output_key)
        
        # Step 4: Save transcription to Firestore
        update_job_status(
            job_id, 
            'completed', 
            outputVideoKey=output_key, 
            progress=100,
            transcription=transcription
        )
        
        # Cleanup
        if os.path.exists(input_path): os.remove(input_path)
        if os.path.exists(output_path): os.remove(output_path)
        
        print(f"✅ Job completed with transcription!")
        return True
    except Exception as e:
        print(f"❌ Failed: {e}")
        import traceback
        traceback.print_exc()
        update_job_status(job_id, 'failed', error=str(e))
        return False

def process_caption_job(job):
    """Process caption generation job."""
    job_id = job['id']
    parent_job_id = job.get('parentJobId')
    caption_settings = job.get('captionSettings', {})
    
    print(f"\n{'='*50}\n🎬 Processing caption job: {job_id}\n{'='*50}")
    
    try:
        update_job_status(job_id, 'processing')
        os.makedirs('/content/temp', exist_ok=True)
        
        # Get parent job for transcription and video
        parent_doc = db.collection('jobs').document(parent_job_id).get()
        if not parent_doc.exists:
            raise Exception(f"Parent job {parent_job_id} not found")
        
        parent_job = parent_doc.to_dict()
        transcription = parent_job.get('transcription')
        output_video_key = parent_job.get('outputVideoKey')
        
        if not transcription or not output_video_key:
            raise Exception("Parent job missing transcription or output video")
        
        # Download the cropped video
        input_path = f'/content/temp/input_{job_id}.mp4'
        download_from_r2(output_video_key, input_path)
        
        # Generate ASS subtitles
        ass_path = f'/content/temp/captions_{job_id}.ass'
        generate_ass_subtitles(transcription, ass_path, caption_settings)
        
        # Burn captions into video
        output_path = f'/content/temp/captioned_{job_id}.mp4'
        burn_captions(input_path, output_path, ass_path)
        
        # Upload captioned video
        captioned_key = f"outputs/captioned_{job_id}.mp4"
        upload_to_r2(output_path, captioned_key)
        
        update_job_status(
            job_id,
            'completed',
            outputVideoKey=captioned_key,
            progress=100
        )
        
        # Cleanup
        if os.path.exists(input_path): os.remove(input_path)
        if os.path.exists(output_path): os.remove(output_path)
        if os.path.exists(ass_path): os.remove(ass_path)
        
        print(f"✅ Caption job completed!")
        return True
    except Exception as e:
        print(f"❌ Failed: {e}")
        import traceback
        traceback.print_exc()
        update_job_status(job_id, 'failed', error=str(e))
        return False

print("✅ Worker functions loaded")

In [ ]:
# Cell 8: Run Worker Loop 🚀
POLL_INTERVAL = 10

print("🚀 Starting Easy-Edits Worker...")
print(f"   YOLO Model: {YOLO_MODEL} (Stabilized Detection)")
print(f"   Whisper Model: {WHISPER_MODEL} (Transcription)")
print(f"   Smoothing: {SMOOTHING_WINDOW_SIZE} frames")
print(f"   Polling every {POLL_INTERVAL}s. Press STOP to stop.\n")

job_count = 0
caption_count = 0

while True:
    try:
        # Check for autocrop jobs
        jobs = get_pending_jobs()
        if jobs:
            print(f"📥 Found {len(jobs)} autocrop job(s)")
            for job in jobs:
                process_job(job)
                job_count += 1
        
        # Check for caption jobs
        caption_jobs = get_caption_jobs()
        if caption_jobs:
            print(f"📥 Found {len(caption_jobs)} caption job(s)")
            for job in caption_jobs:
                process_caption_job(job)
                caption_count += 1
        
        if not jobs and not caption_jobs:
            print(f"💤 No jobs. Autocrop: {job_count}, Captions: {caption_count}. Waiting...")
        
        time.sleep(POLL_INTERVAL)
    except KeyboardInterrupt:
        print("\n⏹️ Stopped")
        break
    except Exception as e:
        print(f"❌ Error: {e}")
        time.sleep(POLL_INTERVAL)